In [1]:
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
import pandas as pd 
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
df = pd.read_csv(r"C:\Users\DELL\OneDrive\Documents\Deep Learning\Titanic Survivor Dataset.csv")
df.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare
0,1,1,3,"Johnson, Mrs. Margaret",female,NaN,1,1,470881,0.78
1,2,0,3,"Brown, Rev. John",male,16.1,1,0,187159,11.65
2,3,1,2,"Brown, Miss. Margaret",female,26.5,0,0,326034,45.68


In [3]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            473
SibSp            0
Parch            0
Ticket           0
Fare             0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df.drop(columns=["Name"], inplace=True)

In [6]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
dtype: object

In [7]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, drop="first")

In [8]:
df["Sex"]  = ohe.fit_transform(df[["Sex"]])

In [9]:
df.drop("Ticket", axis=1, inplace=True)

In [10]:
df.drop(columns=["PassengerId"], inplace=True)

In [12]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
df["Age"] = sc.fit_transform(df[["Age"]])
df["Fare"] = sc.fit_transform(df[["Fare"]])
df["SibSp"] = sc.fit_transform(df[["SibSp"]])
df["Parch"] = sc.fit_transform(df[["Parch"]])

In [13]:
x = df.drop(['Survived'], axis=1)
y = df["Survived"]

In [14]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [15]:
X_train["Age"] = X_train["Age"].fillna(X_train["Age"].median())
X_test["Age"] = X_test["Age"].fillna(X_test["Age"].median())

In [16]:
model = Sequential([Input(shape=(X_train.shape[1], )), 
                    Dense(32, activation='relu'), 
                    Dropout(0.2), 
                    Dense(16, activation='relu'), 
                    Dense(1, activation='sigmoid')
                    ])

In [17]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=["accuracy"])

In [18]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [19]:
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.20, callbacks=early_stopping, verbose=1)

Epoch 1/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.5422 - loss: 0.6870 - val_accuracy: 0.6017 - val_loss: 0.6626
Epoch 2/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5834 - loss: 0.6578 - val_accuracy: 0.6128 - val_loss: 0.6511
Epoch 3/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6329 - loss: 0.6382 - val_accuracy: 0.6017 - val_loss: 0.6460
Epoch 4/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6260 - loss: 0.6336 - val_accuracy: 0.6100 - val_loss: 0.6358
Epoch 5/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6308 - loss: 0.6241 - val_accuracy: 0.6267 - val_loss: 0.6323
Epoch 6/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6615 - loss: 0.6122 - val_accuracy: 0.6128 - val_loss: 0.6319
Epoch 7/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6560 - loss: 0.6117 - val_accuracy: 0.6462 - val_loss: 0.6231
Epoch 8/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6574 - loss: 0.6066 - val_accuracy: 0.6379 - 

In [20]:
y_prob = model.predict(X_test)
y_pred = (y_prob>0.5).astype(int).ravel()

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 


In [21]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print(accuracy_score(y_test, y_pred))

0.6049107142857143


In [22]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.52      0.65      0.58       185
           1       0.70      0.57      0.63       263

    accuracy                           0.60       448
   macro avg       0.61      0.61      0.60       448
weighted avg       0.62      0.60      0.61       448

